# JagoJual — Fine-tune QLoRA (Kaggle)

Notebook siap-jalan untuk melatih adapter LoRA (Qwen2.5-7B) mode Pelatih.

**Sebelum Run All:**
1. Panel kanan → **Accelerator: GPU P100** (atau T4 x2), **Internet: ON**.
2. Repo GitHub sudah berisi **`data/dialogs/` ATAU `model/training/1_generate_local.py`** (+ `scenario_matrix.json`). Kalau dialog belum dipush, sel 3 akan regenerate sendiri dari generator lokal (deterministik).
3. Jalankan berurutan. Untuk latihan panjang tanpa harus jaga tab: **Save Version → Run All (Commit)**.

Output akhir: `model/checkpoints/` (adapter + `training_meta.json` + `eval.json`) → di-zip untuk diunduh dari tab **Output**.


## 1 · Clone repo + install dependency + cek GPU


In [ ]:
!rm -rf /kaggle/working/jagojual
!git clone -q https://github.com/Kenrockender/jagojual-aic.git /kaggle/working/jagojual
%cd /kaggle/working/jagojual/model/training
!pip install -q -r requirements-train.txt
import torch
print('GPU:', torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())

## 2 · Pastikan dataset ada (regenerate kalau belum dipush)
Generator lokal deterministik (seed 42) — hasilnya identik dengan yang di laptop. Aman dijalankan ulang.


In [ ]:
import glob, os
n = len([f for f in glob.glob('../../data/dialogs/*.json') if not f.endswith('schema.json')])
print('dialog terdeteksi:', n)
if n < 50:
    print('Dialog belum lengkap → regenerate dari 1_generate_local.py')
    !python 1_generate_local.py
    # buang duplikat scenario_id yang sudah punya seed manual (contoh_*.json)
    for dup in ['otomotif_boros_bbm_01.json','elektronik_harga_toko_sebelah_01.json']:
        p='../../data/dialogs/'+dup
        if os.path.exists(p) and os.path.exists('../../data/dialogs/contoh_'+dup): os.remove(p)
    n = len([f for f in glob.glob('../../data/dialogs/*.json') if not f.endswith('schema.json')])
print('total dialog siap:', n)

## 3 · Validasi dialog (harus 0 error)


In [ ]:
!python 1_generate_data.py --validate-only | tail -3

## 4 · Bentuk contoh SFT (train / val / test)


In [ ]:
!python 2_prepare_sft.py

## 5 · Fine-tune QLoRA
Default: Qwen2.5-7B-Instruct, 4-bit, LoRA r=16, 3 epoch, batch 1 × grad-accum 8, seq 2048.

- **OOM?** tambahkan `--max-seq-len 1024` (atau `--grad-accum 16`). Terakhir: ganti `--base-model Qwen/Qwen2.5-3B-Instruct` (backend juga harus 3B).
- **Sesi mati di tengah?** jalankan ulang sel ini dengan tambahan `--resume` (checkpoint tiap 100 step).


### 5a · Smoke test (WAJIB, ~2 menit) — pastikan training jalan dulu
Latih 5 step ke folder terpisah. Kalau muncul **`Adapter tersimpan`** tanpa error, berarti aman lanjut full run. Kalau error, berhenti di sini (jangan buang 2 jam).


In [ ]:
!python 3_finetune_qlora.py --train ../../data/sft/train.jsonl --val ../../data/sft/val.jsonl --out ../checkpoints_smoke --max-steps 5 --max-seq-len 512

In [ ]:
!python 3_finetune_qlora.py --train ../../data/sft/train.jsonl --val ../../data/sft/val.jsonl --out ../checkpoints

## 6 · Evaluasi (adapter vs base) — bukti fine-tune berguna
Angka di sini masuk proposal §Metodologi. Cari: **JSON valid** & **macro-F1 teknik** menang telak vs base.


In [ ]:
!python 4_evaluate.py --test ../../data/sft/test.jsonl --adapter ../checkpoints --mode both --out ../checkpoints/eval.json
import json; print(json.dumps(json.load(open('../checkpoints/eval.json')), ensure_ascii=False, indent=2))

## 7 · Kemas adapter untuk diunduh
Zip muncul di tab **Output** (kanan). Unduh → di laptop: taruh isinya ke `model/checkpoints/`, commit, push.


In [ ]:
import shutil, os
os.chdir('/kaggle/working/jagojual')
shutil.make_archive('/kaggle/working/jagojual_adapter', 'zip', 'model/checkpoints')
print('siap diunduh: /kaggle/working/jagojual_adapter.zip')
print(os.listdir('model/checkpoints'))

---
### Setelah unduh (di laptop, di dalam git clone)
```bash
# ekstrak isi zip ke model/checkpoints/
git add model/checkpoints
git commit -m "feat: tambah adapter LoRA hasil fine-tune QLoRA Qwen2.5-7B"
git push
```
Sertakan `training_meta.json` + `eval.json` (bukti angka proposal). Lalu jalankan backend `JAGOJUAL_MODE=local`.
